<a href="https://colab.research.google.com/github/asipnana/ProjectNLP/blob/main/notebooks/04_fasttext.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy==1.26.4

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653906 sha256=e741b6884ecb8bc1916eee924ddd6c36e34cb956eb8b004e80db759a4b54a6fc
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
import fasttext
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)
from sklearn.metrics import classification_report

In [ ]:
train_df = pd.read_csv("train (2).csv")
test_df = pd.read_csv("test (2).csv")

In [ ]:
with open("fasttext_train.txt", "w", encoding="utf-8") as f:
    for _, row in train_df.iterrows():
        label = row['sentiment']
        review = row['clean_review']
        f.write(f"__label__{label} {review}\n")

In [ ]:
model = fasttext.train_supervised(input="fasttext_train.txt", epoch=25, lr=1.0, wordNgrams=2, dim=100)

In [ ]:
y_true = []
y_pred = []

for _, row in test_df.iterrows():
    review = row['clean_review']
    prediction = model.predict(review)
    predicted_label = prediction[0][0]
    predicted_label = predicted_label.replace("__label__", "")
    y_true.append(row['sentiment'])
    y_pred.append(predicted_label)

In [ ]:
accuracy = accuracy_score(y_true, y_pred)
print("Accuracy: ", accuracy)

f1 = f1_score(y_true, y_pred, average="weighted")
print("F1-Score: ", f1)

Accuracy:  0.9396984924623115
F1-Score:  0.9396396878007056


In [ ]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    Negative       0.93      0.96      0.94       413
    Positive       0.95      0.92      0.94       383

    accuracy                           0.94       796
   macro avg       0.94      0.94      0.94       796
weighted avg       0.94      0.94      0.94       796



In [ ]:
results = pd.DataFrame({
    "Model": ["FastText"], "Accuracy": [accuracy_score(y_true, y_pred)],
    "Precision": [precision_score(y_true, y_pred, average="weighted")],
    "Recall": [recall_score(y_true, y_pred, average="weighted")],
    "F1": [f1_score(y_true, y_pred, average="weighted")]
})

print(results)

      Model  Accuracy  Precision    Recall       F1
0  FastText  0.939698   0.940165  0.939698  0.93964


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, log_loss

y_true = []
y_pred_labels = []
y_pred_probs = []  # Menyimpan probabilitas untuk kelas positif

# Definisikan target kelas positif sesuai datasetmu (misal: 'Positive')
# FastText akan mengembalikan probabilitas untuk semua kelas, kita perlu ekstrak salah satu.
POSITIVE_LABEL = "Positive"

for _, row in test_df.iterrows():
    review = str(row['clean_review']).strip() if pd.notna(row['clean_review']) else "kosong"

    # Ambil 2 prediksi teratas beserta probabilitasnya
    predictions, probabilities = model.predict(review, k=2)

    # 1. Bersihkan label untuk prediksi teratas (indeks 0)
    top_label = predictions[0].replace("__label__", "")
    y_pred_labels.append(top_label)

    # 2. Petakan probabilitas ke kelas POSITIVE_LABEL demi ROC-AUC & Log Loss
    prob_dict = {pred.replace("__label__", ""): prob for pred, prob in zip(predictions, probabilities)}

    # Ambil probabilitas untuk kelas positif (jika tidak ada, sisanya adalah 1 - probabilitas kelas lain)
    if POSITIVE_LABEL in prob_dict:
        pos_prob = prob_dict[POSITIVE_LABEL]
    else:
        # Jika kelas positif tidak masuk di pred[0], berarti probabilitasnya adalah (1 - probabilitas kelas lawan)
        other_label = list(prob_dict.keys())[0]
        pos_prob = 1.0 - prob_dict[other_label]

    y_pred_probs.append(pos_prob)
    y_true.append(row['sentiment'])

# Konversi y_true ke biner (0 dan 1) khusus untuk perhitungan ROC-AUC dan Log Loss
y_true_binary = [1 if label == POSITIVE_LABEL else 0 for label in y_true]

# --- HITUNG METRIKS ---
accuracy = accuracy_score(y_true, y_pred_labels)
precision = precision_score(y_true, y_pred_labels, pos_label=POSITIVE_LABEL)
recall = recall_score(y_true, y_pred_labels, pos_label=POSITIVE_LABEL)
f1 = f1_score(y_true, y_pred_labels, pos_label=POSITIVE_LABEL)

# ROC-AUC dan Log Loss membutuhkan probabilitas numerik
roc_auc = roc_auc_score(y_true_binary, y_pred_probs)
logloss = log_loss(y_true_binary, y_pred_probs)

# --- MEMBUAT DATAFRAME SEPERTI INDOBERT ---
fasttext_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Log Loss"],
    "Value": [accuracy, precision, recall, f1, roc_auc, logloss]
})

print("FastText Metrics:")
print(fasttext_results)

FastText Metrics:
      Metric     Value
0   Accuracy  0.939698
1  Precision  0.953930
2     Recall  0.919060
3         F1  0.936170
4    ROC-AUC  0.983819
5   Log Loss  0.262138


In [ ]:
import os

# 1. Paksa bikin foldernya dulu di Google Colab
os.makedirs("/content/ProjectNLP/models/fasttext", exist_ok=True)

# 2. Baru jalankan perintah save model FastText kamu
model.save_model("/content/ProjectNLP/models/fasttext/fasttext_model.bin")

print("Mantap! Model FastText berhasil diselamatkan tanpa eror.")

Mantap! Model FastText berhasil diselamatkan tanpa eror.
